# Họ và Tên
# MSSV

### Câu 1 (2 điểm): Cho một ảnh bất kỳ (tên ảnh do sinh viên tự đặt, ví dụ: `my_image.jpg`) và thực hiện các yêu cầu sau:

* Viết chương trình sử dụng Bilateral filter để làm mịn ảnh. (0.5 điểm)  
* Viết chương trình sử dụng Canny Edge Detection để xác định biên của hình ảnh. (0.5 điểm)  
* Đổi màu ảnh bằng cách hoán đổi kênh màu theo thứ tự (ví dụ: BGR → BRG) và lưu thành tên dạng `[ten_anh]_swapped.jpg`. (0.5 điểm)  
* Chuyển ảnh sang không gian màu YCrCb và tách riêng 3 kênh Y, Cr, Cb, lưu thành ảnh grayscale tương ứng (`[ten_anh]_Y.jpg`, `[ten_anh]_Cr.jpg`, `[ten_anh]_Cb.jpg`). (0.5 điểm)


In [ ]:
import cv2
import numpy as np


image_name = "bird.png"
img = cv2.imread(image_name)

bilateral = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
cv2.imwrite(f"{image_name.split('.')[0]}_bilateral.jpg", bilateral)
edges = cv2.Canny(img, threshold1=100, threshold2=200)
cv2.imwrite(f"{image_name.split('.')[0]}_canny.jpg", edges)
swapped = img.copy()
swapped = img[:, :, [0, 2, 1]] 
cv2.imwrite(f"{image_name.split('.')[0]}_swapped.jpg", swapped)
ycrcb = cv2.cvtColor(img, cv2.COLOR_BGR2YCrCb)
Y, Cr, Cb = cv2.split(ycrcb)
cv2.imwrite(f"{image_name.split('.')[0]}_Y.jpg", Y)
cv2.imwrite(f"{image_name.split('.')[0]}_Cr.jpg", Cr)
cv2.imwrite(f"{image_name.split('.')[0]}_Cb.jpg", Cb)

### Câu 2 (4 điểm) Viết một chương trình Python sử dụng OpenCV để tạo menu tương tác cho phép người dùng chọn các kỹ thuật biến đổi hình học và xử lý ảnh nâng cao từ một danh sách, áp dụng đồng thời cho nhiều ảnh.

### Yêu cầu:

1. Menu gồm:  
* Phóng to ảnh (Zoom bằng resize) (0.5 điểm)  
* Xoay ảnh (góc ngẫu nhiên từ 0–360 độ) (0.5 điểm)  
* Lật ảnh ngang (0.5 điểm)  
* Lật ảnh dọc (0.5 điểm)  
* Cắt ảnh (crop ngẫu nhiên vùng giữa ảnh) (0.5 điểm)  
* Thêm viền (padding màu ngẫu nhiên) (0.5 điểm)

2. Chương trình xử lý đồng thời 3 ảnh bất kỳ do sinh viên tự chọn (có thể chọn bằng đường dẫn file hoặc nhập tên ảnh tùy ý). (0.5 điểm)

3. Phím tương ứng để kích hoạt các phương pháp xử lý:  
* Z: Zoom  
* T: Rotate  
* H: Horizontal Flip  
* V: Vertical Flip  
* C: Crop  
* P: Padding (0.5 điểm)

4. Lưu file kết quả với định dạng: `result_[phương pháp]_[tên ảnh gốc].jpg`  
   Ví dụ: `result_crop_cat.jpg`, `result_rotate_image1.jpg` (0.5 điểm)


In [ ]:
import cv2
import os
import random
def zoom_image(img, scale=1.5):
    h, w = img.shape[:2]
    return cv2.resize(img, (int(w * scale), int(h * scale)))

def rotate_image(img):
    angle = random.randint(0, 360)
    h, w = img.shape[:2]
    center = (w // 2, h // 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(img, matrix, (w, h))

def flip_horizontal(img):
    return cv2.flip(img, 1)

def flip_vertical(img):
    return cv2.flip(img, 0)

def crop_random(img):
    h, w = img.shape[:2]
    crop_h, crop_w = h // 2, w // 2
    start_y = random.randint(0, h - crop_h)
    start_x = random.randint(0, w - crop_w)
    return img[start_y:start_y+crop_h, start_x:start_x+crop_w]

def add_padding(img, padding_size=30):
    color = [random.randint(0, 255) for _ in range(3)]
    return cv2.copyMakeBorder(img, padding_size, padding_size, padding_size, padding_size,
                              borderType=cv2.BORDER_CONSTANT, value=color)

def print_menu():
    print("Z - Zoom")
    print("T - Rotate")
    print("H - Flip Horizontal")
    print("V - Flip Vertical")
    print("C - Crop")
    print("P - Padding")
    print("Q - Thoát chương trình")

def get_method_name(key):
    method_map = {
        "Z": "zoom",
        "T": "rotate",
        "H": "fliph",
        "V": "flipv",
        "C": "crop",
        "P": "padding"
    }
    return method_map.get(key.lower(), "processed")

# Áp dụng xử lý ảnh
def apply_operation(img, key):
    if key == "Z":
        return zoom_image(img)
    elif key == "T":
        return rotate_image(img)
    elif key == "H":
        return flip_horizontal(img)
    elif key == "V":
        return flip_vertical(img)
    elif key == "C":
        return crop_random(img)
    elif key == "P":
        return add_padding(img)
    else:
        return img

folder_path = "New folder"
image_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path)
               if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
selected_imgs = image_files[:3]

print(" Danh sách ảnh trong thư mục:")
for idx, fname in enumerate(selected_imgs):
    print(f"{idx+1}. {os.path.basename(fname)}")


while True:
    print_menu()
    key = input(" Nhập lựa chọn (Z/T/H/V/C/P/Q): ").strip().upper()

    if key == "Q":
        print(" Kết thúc chương trình.")
        break

    method = get_method_name(key)

    for path in selected_imgs:
        img = cv2.imread(path)
        if img is None:
            print(f" Không đọc được ảnh: {path}")
            continue

        processed = apply_operation(img, key)

        base = os.path.basename(path)
        name, _ = os.path.splitext(base)
        output_name = f"result_{method}_{name}.jpg"
        output_path = os.path.join(folder_path, output_name)

        cv2.imwrite(output_path, processed)
        print(f"Đã lưu ảnh: {output_path}")



 Danh sách ảnh trong thư mục:
1. a.jpg
2. butterfly.jpg
3. quang_ninh.jpg

==== MENU XỬ LÝ ẢNH ====
Z - Zoom
T - Rotate
H - Flip Horizontal
V - Flip Vertical
C - Crop
P - Padding
Q - Thoát chương trình
Đã lưu ảnh: New folder\result_processed_a.jpg
Đã lưu ảnh: New folder\result_processed_butterfly.jpg
Đã lưu ảnh: New folder\result_processed_quang_ninh.jpg

==== MENU XỬ LÝ ẢNH ====
Z - Zoom
T - Rotate
H - Flip Horizontal
V - Flip Vertical
C - Crop
P - Padding
Q - Thoát chương trình
 Kết thúc chương trình.


### Câu 3 (4 điểm) Viết một chương trình Python để xử lý 3 ảnh bất kỳ do sinh viên tự chọn.

* Thêm viền đen 20 pixel cho ảnh đầu tiên. (0.5 điểm)  
* Xoay ảnh thứ hai 45 độ và phóng to 1.5 lần. (0.5 điểm)  
* Tăng kích thước ảnh thứ ba lên 4 lần, sau đó áp dụng Bilateral Filter với tham số tùy chọn. (1.5 điểm)  
* Thay đổi độ sáng và độ tương phản ảnh thứ ba theo công thức:

$$
I_{out}(x, y) = \alpha \cdot I_{in}(x, y) + \beta
$$

Trong đó:  

$$
\alpha \in [0.6, 2.0], \quad \beta \in [-60, 60]
$$

Giá trị đầu ra cần được giới hạn trong khoảng [0, 255] bằng công thức:

$$
I_{out}(x, y) = \text{clip}(I_{out}(x, y), 0, 255)
$$


In [ ]:
import cv2
import numpy as np
import os
folder_path = "New folder"
image_files = [f for f in os.listdir(folder_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

if len(image_files) < 3:
    print(" Cần ít nhất 3 ảnh trong thư mục.")
    exit()

img1_path = os.path.join(folder_path, image_files[0])
img2_path = os.path.join(folder_path, image_files[1])
img3_path = os.path.join(folder_path, image_files[2])

img1 = cv2.imread(img1_path)
img2 = cv2.imread(img2_path)
img3 = cv2.imread(img3_path)

img1_padded = cv2.copyMakeBorder(img1, 20, 20, 20, 20, borderType=cv2.BORDER_CONSTANT, value=[0, 0, 0])
cv2.imwrite(os.path.join(folder_path, f"result_padding_{image_files[0]}"), img1_padded)

(h, w) = img2.shape[:2]
center = (w // 2, h // 2)
rotation_matrix = cv2.getRotationMatrix2D(center, 45, 1.5)
rotated_scaled_img = cv2.warpAffine(img2, rotation_matrix, (int(w * 1.5), int(h * 1.5)))
cv2.imwrite(os.path.join(folder_path, f"result_rotate_scale_{image_files[1]}"), rotated_scaled_img)

img3_resized = cv2.resize(img3, (0, 0), fx=4, fy=4)
img3_filtered = cv2.bilateralFilter(img3_resized, d=9, sigmaColor=75, sigmaSpace=75)
alpha = 1.5 
beta = 40    
img3_brightness = cv2.convertScaleAbs(img3_filtered, alpha=alpha, beta=beta)
cv2.imwrite(os.path.join(folder_path, f"result_enhance_{image_files[2]}"), img3_brightness)

print(" Xử lý xong. Đã lưu ảnh kết quả trong thư mục 'New folder'")

 Xử lý xong. Đã lưu ảnh kết quả trong thư mục 'New folder'


# Chúc các bạn thi may mắn và đạt điểm 10.